# M1-INF4 – Python Workshop for Data Analysis
## Chapter 2 – Data Manipulation with pandas

**M1 IBI / SBI — Paris 1 Panthéon-Sorbonne**

### Learning objectives

By the end of this chapter, students will be able to:

- Understand pandas `DataFrame` and `Series` objects
- Inspect and explore a dataset
- Select, filter, and sort data
- Create and transform columns
- Compute summary statistics
- Aggregate data using `groupby()`
- Manipulate indexes and subsets
- Import and export tabular data

## 1. Introducing DataFrames

`pandas` is a Python library designed for data manipulation and analysis.
Its central data structure is the **DataFrame**, a two-dimensional table
organized into rows and columns.

Some of the first methods we use when discovering a dataset are:

- `head()` — display the first rows
- `info()` — inspect columns, data types, and missing values
- `describe()` — compute descriptive statistics

In [1]:
import pandas as pd

data = {
    "Name": ["Fido", "Buddy", "Rex"],
    "Age": [4, 5, 6]
}

df = pd.DataFrame(data)

df.head()

,Name,Age
0,Fido,4
1,Buddy,5
2,Rex,6


### From NumPy to pandas
A NumPy array gives you numerical operations; pandas adds **column names, row labels and heterogeneous column types**.
A `Series` is one labelled column. A `DataFrame` is a table containing several columns.
Run the examples, then complete the exercises in the empty cells. Explain what each result means, not only which method you used.

In [2]:
print(type(df["Age"]))
print(type(df[["Age"]]))
df[["Name", "Age"]]

<class 'pandas.Series'>
<class 'pandas.DataFrame'>


,Name,Age
0,Fido,4
1,Buddy,5
2,Rex,6


## 2. Inspecting real data
We start with `homelessness.csv`, supplied with DataCamp's *Data Manipulation with pandas* course.
Each row represents one US state or the District of Columbia. The file contains counts of individuals experiencing homelessness, people in families experiencing homelessness, state population and region.
These are observations in a supplied dataset, **not current estimates**. An absolute count and a rate answer different questions.

Keep the `notebooks/` and `data/` folders together. Open this notebook from `notebooks/`.

In [3]:
import pandas as pd
homelessness = pd.read_csv("../data/homelessness.csv")

homelessness.head()

,Unnamed: 0,region,state,individuals,family_members,state_pop
0,0,East South Central,Alabama,2570.0,864.0,4887681
1,1,Pacific,Alaska,1434.0,582.0,735139
2,2,Mountain,Arizona,7259.0,2606.0,7158024
3,3,West South Central,Arkansas,2280.0,432.0,3009733
4,4,Pacific,California,109008.0,20964.0,39461588


The CSV also contains an exported index column (`Unnamed: 0`). We remove this technical column explicitly; it is not a measured variable.

In [4]:
homelessness = homelessness.drop(columns=["Unnamed: 0"], errors="ignore")
print("Shape:", homelessness.shape)
print("Columns:", homelessness.columns.tolist())
homelessness.info()

Shape: (51, 5)
Columns: ['region', 'state', 'individuals', 'family_members', 'state_pop']
<class 'pandas.DataFrame'>
RangeIndex: 51 entries, 0 to 50
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   region          51 non-null     str    
 1   state           51 non-null     str    
 2   individuals     51 non-null     float64
 3   family_members  51 non-null     float64
 4   state_pop       51 non-null     int64  
dtypes: float64(2), int64(1), str(2)
memory usage: 2.1 KB


In [5]:
homelessness.describe()

,individuals,family_members,state_pop
count,51.000000,51.000000,5.100000e+01
mean,7225.784314,3504.882353,6.405637e+06
std,15991.025083,7805.411811,7.327258e+06
min,434.000000,75.000000,5.776010e+05
25%,1446.500000,592.000000,1.777414e+06
50%,3082.000000,1482.000000,4.461153e+06
75%,6781.500000,3196.000000,7.340946e+06
max,109008.000000,52070.000000,3.946159e+07


### Your turn — Understand the table
1. Display the last three rows.
2. Count the missing values in each column.
3. How many distinct regions are represented?
4. In a Markdown cell, explain what one row represents and why `state_pop` should not be added to the homelessness counts.

In [6]:
homelessness.tail(3)


,region,state,individuals,family_members,state_pop
48,South Atlantic,West Virginia,1021.0,222.0,1804291
49,East North Central,Wisconsin,2740.0,2167.0,5807406
50,Mountain,Wyoming,434.0,205.0,577601


## 3. Selecting, filtering and sorting
Select a column with `table["column"]`; select several columns with a list of names.
A comparison creates a Boolean mask, just as with NumPy. `.loc[rows, columns]` lets you choose both rows and columns.

In [7]:
homelessness[["state", "individuals"]].head()

,state,individuals
0,Alabama,2570.0
1,Alaska,1434.0
2,Arizona,7259.0
3,Arkansas,2280.0
4,California,109008.0


In [8]:
mask = homelessness["region"] == "Mountain"
homelessness.loc[mask, ["state", "individuals", "state_pop"]]

,state,individuals,state_pop
2,Arizona,7259.0,7158024
5,Colorado,7607.0,5691287
12,Idaho,1297.0,1750536
26,Montana,983.0,1060665
28,Nevada,7058.0,3027341
31,New Mexico,1949.0,2092741
44,Utah,1904.0,3153550
50,Wyoming,434.0,577601


Combine conditions with `&` (and), `|` (or), and `~` (not). Put each comparison in parentheses; do not use Python's `and` between Series.

In [9]:
homelessness.loc[
    (homelessness["region"] == "Mountain") & (homelessness["individuals"] > 1000),
    ["state", "individuals"]
]

,state,individuals
2,Arizona,7259.0
5,Colorado,7607.0
12,Idaho,1297.0
28,Nevada,7058.0
31,New Mexico,1949.0
44,Utah,1904.0


In [10]:
homelessness.sort_values("individuals", ascending=False)[["state", "individuals"]].head(5)

,state,individuals
4,California,109008.0
32,New York,39827.0
9,Florida,21443.0
43,Texas,19199.0
47,Washington,16424.0


`head(5)` keeps the first five rows of the current order. `.iloc` selects by position; `.loc` selects by labels or Boolean masks. Neither sorting nor selecting changes the original table unless you assign the result.

In [11]:
homelessness.iloc[:3, :2]

,region,state
0,East South Central,Alabama
1,Pacific,Alaska
2,Mountain,Arizona


### Your turn — A targeted selection
Select states with more than 5 million inhabitants and more than 5,000 individuals experiencing homelessness.
Display only `state`, `state_pop` and `individuals`, ordered by `individuals` from largest to smallest.
Save your result in a variable called `selected_states`.

In [12]:
selected_states = homelessness.loc[
    (homelessness["state_pop"] > 5_000_000)
    & (homelessness["individuals"] > 5_000),
    ["state", "state_pop", "individuals"]
].sort_values("individuals", ascending=False)

selected_states

,state,state_pop,individuals
4,California,39461588,109008.0
32,New York,19530351,39827.0
9,Florida,21244317,21443.0
43,Texas,28628666,19199.0
47,Washington,7523869,16424.0
38,Pennsylvania,12800922,8163.0
5,Colorado,5691287,7607.0
2,Arizona,7158024,7259.0
10,Georgia,10511131,6943.0
35,Ohio,11676341,6929.0


## 4. Creating meaningful indicators
Vectorized calculations work on entire columns. There is no need to loop over rows.
Here is a rate for **individuals only**; people in families are not included in this indicator.

In [13]:
homelessness["individuals_per_10k"] = homelessness["individuals"] / homelessness["state_pop"] * 10_000
homelessness[["state", "individuals_per_10k"]].head().round(2)

,state,individuals_per_10k
0,Alabama,5.26
1,Alaska,19.51
2,Arizona,10.14
3,Arkansas,7.58
4,California,27.62


### Group problem 1 — Where should an analyst look first?
A public-policy team asks: **“Which states have the greatest need?”** Your group must show why the answer depends on the indicator.

1. On a copy named `needs`, create `total_homeless` by adding `individuals` and `family_members`.
2. Create `total_per_10k`, using the state population as denominator.
3. Build two top-five tables: one by total count and one by rate. Include state names and the indicator values.
4. Identify the states common to both lists (manual comparison is acceptable).
5. Write a short recommendation: which ranking answers a question about the number of people to support? Which answers a question about prevalence? Explain one limitation of using these figures alone to allocate funding.

**Work as a group.** Agree on your indicators before coding. Each member must be able to explain the denominator and reproduce the filtering/sorting steps. Keep your code and a three-sentence interpretation below. No solution is included in this notebook.

In [14]:
#1
needs = homelessness.copy()
needs["total_homeless"] = needs["individuals"] + needs["family_members"]
#2
needs["total_per_10k"] = needs["total_homeless"] / needs["state_pop"] * 10_000
#3
top5_count = (needs[["state", "total_homeless"]].sort_values("total_homeless", ascending=False).head(5))
print(top5_count)
top5_rate = (needs[["state", "total_per_10k"]].sort_values("total_per_10k", ascending=False).head(5))
print(top5_rate)

         state  total_homeless
4   California        129972.0
32    New York         91897.0
9      Florida         31030.0
43       Texas         25310.0
47  Washington         22304.0
                   state  total_per_10k
8   District of Columbia      98.411083
32              New York      47.053430
11                Hawaii      45.966720
37                Oregon      34.615960
4             California      32.936333


**Group interpretation:**

*Write your recommendation here.*

## 5. Summarising data with pandas
A summary reduces many observations to a few numbers. Choose the variable and the question first.
`count()` counts non-missing values; `size()` counts rows. `value_counts()` counts occurrences of each category.

In [15]:
homelessness["individuals"].agg(["sum", "mean", "median", "max"])

sum       368515.000000
mean        7225.784314
median      3082.000000
max       109008.000000
Name: individuals, dtype: float64

In [16]:
homelessness["region"].value_counts()

region
South Atlantic        9
Mountain              8
West North Central    7
New England           6
Pacific               5
East North Central    5
East South Central    4
West South Central    4
Mid-Atlantic          3
Name: count, dtype: int64

### From a single summary to one summary per group
`groupby()` follows three steps: split rows into groups, apply a calculation, and combine the results.
The following example sums counts within each region. It does not average state rates.

In [17]:
regional = homelessness.groupby("region")[["individuals", "family_members", "state_pop"]].sum()
regional

,individuals,family_members,state_pop
region,,,
East North Central,25406.0,14002.0,46886387
East South Central,12468.0,3889.0,19101485
Mid-Atlantic,54038.0,60769.0,41217298
Mountain,28491.0,9258.0,24511745
New England,12903.0,17499.0,14829322
Pacific,142136.0,33162.0,53323075
South Atlantic,52260.0,23818.0,65229624
West North Central,13971.0,8242.0,21350241
West South Central,26842.0,8110.0,40238324


The group labels now form the index. `reset_index()` turns those labels back into a regular column.

In [18]:
regional = regional.reset_index()
regional.head()

,region,individuals,family_members,state_pop
0,East North Central,25406.0,14002.0,46886387
1,East South Central,12468.0,3889.0,19101485
2,Mid-Atlantic,54038.0,60769.0,41217298
3,Mountain,28491.0,9258.0,24511745
4,New England,12903.0,17499.0,14829322


### Your turn — Compare regions fairly
Using `regional`, create the total homelessness count and the total rate per 10,000 inhabitants for each region.
Sort by rate, highest first. Explain why you divide the **sum of counts by the sum of populations**, rather than taking the simple average of state rates.

In [19]:
regional["total_homeless"] = regional["individuals"] + regional["family_members"]

regional["total_per_10k"] = (
    regional["total_homeless"] / regional["state_pop"] * 10_000
)

regional = regional.sort_values("total_per_10k", ascending=False)

regional[["region", "total_homeless", "total_per_10k"]]

,region,total_homeless,total_per_10k
5,Pacific,175298.0,32.874698
2,Mid-Atlantic,114807.0,27.854082
4,New England,30402.0,20.501274
3,Mountain,37749.0,15.400372
6,South Atlantic,76078.0,11.663106
7,West North Central,22213.0,10.404098
8,West South Central,34952.0,8.686246
1,East South Central,16357.0,8.563209
0,East North Central,39408.0,8.404998


## 6. Transfer your skills to a business dataset
The second file, `sales_subset.csv`, is a sample supplied with the same DataCamp course.
One row describes a store–department–date observation; `weekly_sales` records weekly sales in US dollars. The file is a subset: do not assume complete time coverage or equal numbers of observations per store.

Main columns: `store` (store identifier), `type` (store category), `department`, `date`, `weekly_sales`, `is_holiday`. Other columns describe temperature, fuel price and unemployment.
Negative sales occur in the file. Their cause is not established here: keep them for this exercise and report their presence rather than silently removing them.

In [20]:
sales = pd.read_csv("../data/sales_subset.csv", index_col=0)
sales.head()

,store,type,department,date,weekly_sales,is_holiday,temperature_c,fuel_price_usd_per_l,unemployment
0,1,A,1,2010-02-05,24924.50,False,5.727778,0.679451,8.106
1,1,A,1,2010-03-05,21827.90,False,8.055556,0.693452,8.106
2,1,A,1,2010-04-02,57258.43,False,16.816667,0.718284,7.808
3,1,A,1,2010-05-07,17413.94,False,22.527778,0.748928,7.808
4,1,A,1,2010-06-04,17558.09,False,27.050000,0.714586,7.808


### Group problem 2 — Does the biggest total mean the best performance?
A retail manager claims: **“The store type with the highest total sales performs best.”** Investigate this claim using the supplied sample.

1. Inspect the table: report its dimensions, number of distinct stores and number of observations with negative `weekly_sales`.
2. For each store type, calculate total sales, mean sales per observation and number of observations. You may use separate `groupby()` operations or `.agg()`.
3. Determine whether the same store type leads for total and mean sales.
4. For each store, calculate total sales and number of observations. Display the five stores with the largest totals.
5. Write a four-sentence response to the manager. Explain how unequal sample sizes affect totals and why a mean per observation is not a mean per store or proof of profitability.
6. Save the store-type summary as `sales_by_type.csv` in the notebook folder, with `index=False` (after `reset_index()` if necessary).

**Group deliverable:** reproducible code, the summary tables and your written response. Split the work, then review the whole answer together. Each member must be able to explain all calculations. No solution is included here.

In [21]:
sales = pd.read_csv("../data/sales_subset.csv", index_col=0)
#1
print("Size:", sales.shape)
print("Nuber of storez:", sales["store"].nunique())
n_negative = (sales["weekly_sales"] < 0).sum()
print("Observations with negative sales:", n_negative)

Size: (10774, 9)
Nuber of storez: 12
Observations with negative sales: 108


In [22]:
sales_by_type = sales.groupby("type").agg(
    total_sales=("weekly_sales", "sum"), 
    mean_sales=("weekly_sales", "mean"), 
    n_obs=("weekly_sales", "count"))
sales_by_type

,total_sales,mean_sales,n_obs
type,,,
A,2.337163e+08,23674.667242,9872
B,2.317840e+07,25696.678370,902


In [23]:
leader_total = sales_by_type["total_sales"].idxmax()
leader_mean = sales_by_type["mean_sales"].idxmax()
print(leader_total)
print(leader_mean)
leader_total==leader_mean

A
B


False

In [24]:
sales_by_store = sales.groupby("store").agg(
    total_sales=("weekly_sales", "sum"),
    n_obs=("weekly_sales", "count"),
).sort_values("total_sales", ascending=False)

sales_by_store.head(5)

,total_sales,n_obs
store,,
14,26889842.67,885
20,25828317.41,910
2,23786139.34,897
4,23540414.45,901
13,23431368.47,913


In [25]:
sales_by_type.reset_index().to_csv("../data/sales_by_type.csv", index=False)

**Group interpretation:**

*Write your response to the manager here.*

## 7. Check your understanding
Before moving on, make sure you can:
- explain the difference between a Series and a DataFrame;
- select columns and combine filtering conditions;
- sort and build a vectorized indicator;
- distinguish a count, a sum, a mean and a rate;
- use `groupby()` and export a table;
- explain the unit represented by a row before interpreting results.

Restart the kernel and run the notebook from top to bottom. Your results should not depend on running cells in a special order.

**Next:** investigate missing values, inconsistent types and duplicates more systematically.

### Sources and teaching adaptation
Datasets: DataCamp, *Data Manipulation with pandas*, supplied course-resource files `homelessness.csv` and `sales_subset.csv`.
This notebook extends the existing M1-INF4 pandas introduction. Its question-led group activities draw on the teaching approach of Anthony's previous course materials. The activities and explanations added here are an original adaptation, not a reproduction of the paid DataCamp exercises or the private cinema dataset.